# Aspect-Conditioned Sentiment Model Training on AspectEngine Predictions (2,000 Reviews)
### Notebook: `predicted_set_based_training.ipynb`

This notebook trains classical Machine Learning sentiment classifiers conditioned on the **aspects predicted by the AspectEngine** (`final_aspect_evaluation/predicted_aspects_2000.csv` / `aspect_predictions_2000_calibrated.csv`) across the 2,000 reviews dataset ($N=11,631$ clauses).

#### Pipeline Overview:
1. **Aspect Conditioning:** Input representation is formatted as:
   $$\text{Input} = \text{"Aspect: "} + \text{Predicted\_Aspect} + \text{" [SEP] "} + \text{Clause\_Text}$$
   Where $\text{Predicted\_Aspect}$ is generated directly by the hybrid AspectEngine rather than oracle ground-truth labels.
2. **Models Trained (Classical ML Only):**
   - **Logistic Regression** (L2 regularized, balanced class weighting)
   - **Linear Support Vector Classifier** (LinearSVC with squared hinge loss)
   - **Random Forest Classifier** (200 trees, balanced sub-sampling)
   - **XGBoost Classifier** (Gradient-boosted decision trees with sample re-weighting)
3. **Partitioning:** 70% Train / 15% Validation / 15% Test grouped strictly by `review_id` (0 reviewer leakage across splits).
4. **Metrics:** Accuracy, Macro-F1, Weighted-F1, and Per-Class F1 (Negative, Neutral, Positive).
5. **Artifacts:** Checkpoints saved to `outputs/predicted_set_based_training/`.


In [ ]:
import os
import sys
import gc
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Dynamic project root detection
curr = os.path.abspath(os.getcwd())
while curr != os.path.dirname(curr):
    if os.path.exists(os.path.join(curr, 'final_aspect_evaluation')) and os.path.exists(os.path.join(curr, 'data')):
        PROJECT_ROOT = curr
        break
    curr = os.path.dirname(curr)
else:
    PROJECT_ROOT = os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd()

# Data Paths with fallback resolution
PRED_ASPECT_CANDIDATES = [
    os.path.join(PROJECT_ROOT, 'final_aspect_evaluation', 'predicted_aspects_2000.csv'),
    os.path.join(PROJECT_ROOT, 'aspect_evaluation', 'aspect_predictions_2000_calibrated.csv'),
    os.path.join(PROJECT_ROOT, 'aspect_evaluation', 'aspect_predictions_2000.csv')
]

ANNOTATION_CANDIDATES = [
    os.path.join(PROJECT_ROOT, 'final_aspect_evaluation', 'modified_annotations_2000.csv'),
    os.path.join(PROJECT_ROOT, 'final_aspect_evaluation', 'modified_annotations_2000_corrected.csv'),
    os.path.join(PROJECT_ROOT, 'annotations', 'combined_2000', 'annotations_2000_final.csv')
]

GOLD_BENCHMARK_CANDIDATES = [
    os.path.join(PROJECT_ROOT, 'data', 'gold', 'gold_aspect_annotated_650_candidate_updated.csv'),
    os.path.join(PROJECT_ROOT, 'data', 'gold', 'gold_aspect_annotated_650_candidate.csv')
]

PRED_ASPECT_PATH = next((p for p in PRED_ASPECT_CANDIDATES if os.path.exists(p)), None)
ANNOTATION_PATH = next((p for p in ANNOTATION_CANDIDATES if os.path.exists(p)), None)
GOLD_PATH = next((p for p in GOLD_BENCHMARK_CANDIDATES if os.path.exists(p)), None)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'predicted_set_based_training')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project Root:               ", PROJECT_ROOT)
print("AspectEngine Predictions:   ", PRED_ASPECT_PATH)
print("Ground-Truth Annotations:   ", ANNOTATION_PATH)
print("Gold Benchmark Dataset:     ", GOLD_PATH)
print("Artifacts Output Directory: ", OUTPUT_DIR)

assert PRED_ASPECT_PATH is not None, "AspectEngine predictions file not found!"
assert ANNOTATION_PATH is not None, "Ground truth annotations file not found!"

## 1. Dataset Ingestion & Predicted-Aspect Conditioning

We align ground-truth sentiment labels from `modified_annotations_2000.csv` with the AspectEngine's predicted aspects from `predicted_aspects_2000.csv`.
- Each clause is assigned its ground-truth sentiment target.
- For each clause, input text is conditioned on its **AspectEngine predicted aspect(s)**:
  `Aspect: <predicted_aspect> [SEP] <clause_text>`
- If no aspect was detected by the engine, it defaults to the catch-all `General Experience` category.


In [ ]:
df_ann = pd.read_csv(ANNOTATION_PATH)
df_pred = pd.read_csv(PRED_ASPECT_PATH)

VALID_SENTIMENTS = ['Positive', 'Negative', 'Neutral']
LABEL_MAP = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
INV_LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

# Filter annotations with valid sentiment
ann_elig = df_ann[df_ann['sentiment'].isin(VALID_SENTIMENTS)].copy()

# Deduplicate to unique clause level with canonical sentiment
clause_sent = ann_elig.groupby(['clause_id', 'review_id', 'clause_text', 'star_rating'])['sentiment'].first().reset_index()

# Merge with AspectEngine predictions
merged = clause_sent.merge(
    df_pred[['clause_id', 'predicted_aspect_labels', 'predicted_aspects_display']],
    on='clause_id',
    how='left'
)

def parse_predicted_aspects(x):
    try:
        res = json.loads(x) if isinstance(x, str) else []
        return res if (isinstance(res, list) and len(res) > 0) else ['General Experience']
    except Exception:
        return ['General Experience']

merged['parsed_aspects'] = merged['predicted_aspect_labels'].apply(parse_predicted_aspects)

# Expand multi-aspect predictions into individual conditioned training instances
expanded_records = []
for _, row in merged.iterrows():
    c_text = str(row['clause_text']).strip()
    s_label = row['sentiment']
    for asp in row['parsed_aspects']:
        expanded_records.append({
            'review_id': row['review_id'],
            'clause_id': row['clause_id'],
            'star_rating': row['star_rating'],
            'predicted_aspect': asp,
            'sentiment': s_label,
            'aspect_conditioned_text': f"Aspect: {asp} [SEP] {c_text}"
        })

df_dataset = pd.DataFrame(expanded_records)

print(f"Total Unique Annotated Clauses:    {len(clause_sent):,}")
print(f"Total Conditioned Instances:       {len(df_dataset):,}")
print(f"Unique Reviews Represented:        {df_dataset['review_id'].nunique():,}")

print("\n--- Sentiment Target Distribution ---")
print(df_dataset['sentiment'].value_counts())

print("\n--- Predicted Aspect Distribution ---")
print(df_dataset['predicted_aspect'].value_counts())

df_dataset[['clause_id', 'predicted_aspect', 'sentiment', 'aspect_conditioned_text']].head(5)

## 2. Grouped Stratified Partitioning (70% Train / 15% Val / 15% Test)

To prevent data leakage, instances are grouped strictly by `review_id` with random seed `42`. No reviewer's text or stylistic cues can appear across both train and test splits.


In [ ]:
# Determine dominant sentiment per review for stratification
review_strat = df_dataset.groupby('review_id')['sentiment'].apply(lambda s: s.value_counts().index[0])

reviews_by_strat = {}
for rid, sent in review_strat.items():
    reviews_by_strat.setdefault(sent, []).append(rid)

np.random.seed(42)
train_rids, val_rids, test_rids = [], [], []

for sent, rids in reviews_by_strat.items():
    shuffled = np.random.permutation(rids)
    n = len(shuffled)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    train_rids.extend(shuffled[:n_train])
    val_rids.extend(shuffled[n_train:n_train + n_val])
    test_rids.extend(shuffled[n_train + n_val:])

train_df = df_dataset[df_dataset['review_id'].isin(train_rids)].copy()
val_df = df_dataset[df_dataset['review_id'].isin(val_rids)].copy()
test_df = df_dataset[df_dataset['review_id'].isin(test_rids)].copy()

# Zero leakage validation
assert len(set(train_rids).intersection(set(val_rids))) == 0, "Leakage between train and val!"
assert len(set(train_rids).intersection(set(test_rids))) == 0, "Leakage between train and test!"
assert len(set(val_rids).intersection(set(test_rids))) == 0, "Leakage between val and test!"

y_train = train_df['sentiment'].map(LABEL_MAP).values
y_val = val_df['sentiment'].map(LABEL_MAP).values
y_test = test_df['sentiment'].map(LABEL_MAP).values

print("=== PARTITIONING SUMMARY (0 REVIEWER LEAKAGE) ===")
print(f"Train Set:      {len(train_df):,} rows ({len(train_rids)} reviews, {len(train_df)/len(df_dataset)*100:.1f}%)")
print(f"Validation Set: {len(val_df):,} rows ({len(val_rids)} reviews, {len(val_df)/len(df_dataset)*100:.1f}%)")
print(f"Test Set:       {len(test_df):,} rows ({len(test_rids)} reviews, {len(test_df)/len(df_dataset)*100:.1f}%)")

# Save split datasets
train_df.to_csv(os.path.join(OUTPUT_DIR, 'train.csv'), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, 'validation.csv'), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, 'test.csv'), index=False)

split_manifest = {
    'train_reviews': len(train_rids),
    'val_reviews': len(val_rids),
    'test_reviews': len(test_rids),
    'train_rows': len(train_df),
    'val_rows': len(val_df),
    'test_rows': len(test_df),
    'label_map': LABEL_MAP
}
with open(os.path.join(OUTPUT_DIR, 'split_manifest.json'), 'w') as f:
    json.dump(split_manifest, f, indent=2)

print("Split manifests and CSVs successfully persisted to:", OUTPUT_DIR)

## 3. TF-IDF Feature Extraction for Classical Models

Features are extracted using word unigrams and bigrams (`max_features=5000`, `sublinear_tf=True`, `min_df=2`).
Vectorization is fitted **strictly on the training split** to guarantee inductive generalization.


In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    sublinear_tf=True,
    min_df=2
)

X_train = tfidf.fit_transform(train_df['aspect_conditioned_text'])
X_val = tfidf.transform(val_df['aspect_conditioned_text'])
X_test = tfidf.transform(test_df['aspect_conditioned_text'])

tfidf_path = os.path.join(OUTPUT_DIR, 'tfidf_vectorizer.joblib')
joblib.dump(tfidf, tfidf_path)

print(f"TF-IDF Vocabulary Size: {len(tfidf.vocabulary_):,} features")
print(f"X_train Shape:         {X_train.shape}")
print(f"X_val Shape:           {X_val.shape}")
print(f"X_test Shape:          {X_test.shape}")
print(f"Saved TF-IDF model to: {tfidf_path}")

## 4. Train 4 Classical ML Models & Persist Checkpoints

We train four distinct classical machine learning algorithms:
1. **Logistic Regression** (L2 regularized, balanced class weighting)
2. **Linear Support Vector Classifier (LinearSVC)**
3. **Random Forest Classifier** (200 estimators)
4. **XGBoost Classifier** (Gradient-boosted softprob multi-class classification)

All models are checkpointed to disk using `joblib`.


In [ ]:
print("="*60)
print("TRAINING 4 CLASSICAL ML MODELS ON PREDICTED-ASPECT DATASET")
print("="*60)

# 1. Logistic Regression
print("\n1. Training Logistic Regression...")
lr = LogisticRegression(C=5.0, class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_path = os.path.join(OUTPUT_DIR, 'logistic_regression.joblib')
joblib.dump(lr, lr_path)
print(f"   Saved: {lr_path}")

# 2. Linear SVM
print("\n2. Training Linear SVM...")
svm = LinearSVC(C=0.2, class_weight='balanced', max_iter=2000, random_state=42)
svm.fit(X_train, y_train)
svm_path = os.path.join(OUTPUT_DIR, 'linear_svm.joblib')
joblib.dump(svm, svm_path)
print(f"   Saved: {svm_path}")

# 3. Random Forest
print("\n3. Training Random Forest...")
rf = RandomForestClassifier(n_estimators=200, max_depth=20, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_path = os.path.join(OUTPUT_DIR, 'random_forest.joblib')
joblib.dump(rf, rf_path)
print(f"   Saved: {rf_path}")

# 4. XGBoost
print("\n4. Training XGBoost Classifier...")
sample_weights_train = compute_sample_weight('balanced', y_train)
xgb = XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    objective='multi:softprob'
)
xgb.fit(X_train, y_train, sample_weight=sample_weights_train)
xgb_path = os.path.join(OUTPUT_DIR, 'xgboost.joblib')
joblib.dump(xgb, xgb_path)
print(f"   Saved: {xgb_path}")

print("\nAll 4 models successfully trained and persisted to disk!")

## 5. Reload Models from Disk & Evaluate on Test Set

To ensure reproducibility and model portability, we delete in-memory models and reload the trained checkpoints from disk for evaluation.


In [ ]:
# Clean in-memory models and recall from disk
del lr, svm, rf, xgb
gc.collect()

print("Recalling models from disk checkpoints...")
reloaded_models = {
    'Logistic Regression': joblib.load(os.path.join(OUTPUT_DIR, 'logistic_regression.joblib')),
    'Linear SVM':          joblib.load(os.path.join(OUTPUT_DIR, 'linear_svm.joblib')),
    'Random Forest':       joblib.load(os.path.join(OUTPUT_DIR, 'random_forest.joblib')),
    'XGBoost':             joblib.load(os.path.join(OUTPUT_DIR, 'xgboost.joblib'))
}

target_names = ['Negative', 'Neutral', 'Positive']
model_names = list(reloaded_models.keys())

test_preds = {}
test_results = []

for name, model in reloaded_models.items():
    preds = model.predict(X_test)
    test_preds[name] = preds
    
    acc = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average='macro')
    weighted_f1 = f1_score(y_test, preds, average='weighted')
    class_f1 = f1_score(y_test, preds, average=None)
    
    test_results.append({
        'Model': name,
        'Accuracy (%)': round(acc * 100, 2),
        'Macro-F1 (%)': round(macro_f1 * 100, 2),
        'Weighted-F1 (%)': round(weighted_f1 * 100, 2),
        'Negative F1 (%)': round(class_f1[0] * 100, 2),
        'Neutral F1 (%)': round(class_f1[1] * 100, 2),
        'Positive F1 (%)': round(class_f1[2] * 100, 2)
    })
    
    print(f"\n==================== {name.upper()} TEST REPORT ====================")
    print(classification_report(y_test, preds, target_names=target_names, digits=4))

df_test_summary = pd.DataFrame(test_results)
df_test_summary.to_csv(os.path.join(OUTPUT_DIR, 'metrics_comparison.csv'), index=False)
with open(os.path.join(OUTPUT_DIR, 'metrics_comparison.json'), 'w') as f:
    json.dump(test_results, f, indent=2)

print("\n===================== TEST SPLIT BENCHMARK SUMMARY =====================")
print(df_test_summary.to_string(index=False))

## 6. Comprehensive Visualizations

We generate:
1. **Four-Panel Normalized Confusion Matrices** displaying class recall alongside Macro-F1 and Weighted-F1 in each title.
2. **Model Performance Comparison Bar Chart** (Accuracy, Macro-F1, Weighted-F1 with direct percentage annotations).
3. **Per-Class F1 Score Comparison** highlighting class-level behavior (Negative, Neutral, Positive).


In [ ]:
# 1. Four-Panel Normalized Confusion Matrices
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))

for idx, name in enumerate(model_names):
    cm = confusion_matrix(y_test, test_preds[name], labels=[0, 1, 2])
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', cbar=False, ax=axes[idx],
                xticklabels=['Neg', 'Neu', 'Pos'], yticklabels=['Neg', 'Neu', 'Pos'], vmin=0, vmax=1)
    axes[idx].set_title(
        f"{name}\nMacro-F1: {test_results[idx]['Macro-F1 (%)']}%\nWeighted-F1: {test_results[idx]['Weighted-F1 (%)']}%",
        fontsize=10, fontweight='bold', pad=10
    )
    axes[idx].set_xlabel('Predicted Label', fontweight='bold')
    if idx == 0:
        axes[idx].set_ylabel('True Label', fontweight='bold')
    else:
        axes[idx].set_ylabel('')

plt.suptitle("Normalized Confusion Matrices on Test Split (Predicted-Set Training)", fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices.png'), dpi=300, bbox_inches='tight')
plt.show()

# 2. Performance Comparison Bar Chart (Accuracy, Macro-F1, Weighted-F1)
fig, ax = plt.subplots(figsize=(11, 4.8))
x = np.arange(len(model_names))
w = 0.25

b1 = ax.bar(x - w, [r['Accuracy (%)'] for r in test_results], w, label='Accuracy (%)', color='#3498db', edgecolor='#333333')
b2 = ax.bar(x, [r['Macro-F1 (%)'] for r in test_results], w, label='Macro-F1 (%)', color='#e67e22', edgecolor='#333333')
b3 = ax.bar(x + w, [r['Weighted-F1 (%)'] for r in test_results], w, label='Weighted-F1 (%)', color='#9b59b6', edgecolor='#333333')

for bar_group in [b1, b2, b3]:
    for bar in bar_group:
        h = bar.get_height()
        ax.annotate(f"{h:.1f}%",
                    xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontweight='bold', fontsize=10)
ax.set_ylabel('Score (%)', fontweight='bold', fontsize=11)
ax.set_title('Test Set Performance: Accuracy vs. Macro-F1 vs. Weighted-F1 (ML Models)', fontweight='bold', fontsize=12, pad=12)
ax.set_ylim(0, 100)
ax.legend(frameon=True, facecolor='white', framealpha=0.9, loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison_chart.png'), dpi=300, bbox_inches='tight')
plt.show()

# 3. Per-Class F1 Score Comparison
fig, ax = plt.subplots(figsize=(11, 4.5))
w = 0.25
ax.bar(x - w, [r['Negative F1 (%)'] for r in test_results], w, label='Negative F1', color='#e74c3c', edgecolor='#333333')
ax.bar(x, [r['Neutral F1 (%)'] for r in test_results], w, label='Neutral F1', color='#95a5a6', edgecolor='#333333')
ax.bar(x + w, [r['Positive F1 (%)'] for r in test_results], w, label='Positive F1', color='#2ecc71', edgecolor='#333333')

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontweight='bold', fontsize=10)
ax.set_ylabel('F1 Score (%)', fontweight='bold', fontsize=11)
ax.set_title('Per-Class F1 Scores on Test Split', fontweight='bold', fontsize=12, pad=12)
ax.set_ylim(0, 100)
ax.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'per_class_f1_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

## 7. External Benchmark: Gold 650 Dataset Evaluation

We evaluate the models trained on the AspectEngine predicted dataset directly on the independent **Gold 650 benchmark dataset** (`gold_aspect_annotated_650_candidate_updated.csv`), observing how models trained under predicted-aspect distributions transfer to gold review clauses.


In [ ]:
if GOLD_PATH and os.path.exists(GOLD_PATH):
    print(f"Evaluating on Gold Standard Benchmark: {GOLD_PATH}")
    gold_df = pd.read_csv(GOLD_PATH)
    
    # Map Gold aspect taxonomy to canonical taxonomy
    aspect_map = {
        'Food': 'Food',
        'Service': 'Service',
        'Ambience': 'Ambience',
        'Price': 'Price / Value',
        'Price / Value': 'Price / Value',
        'General': 'General Experience',
        'General Experience': 'General Experience'
    }
    gold_df['canonical_aspect'] = gold_df['gold_aspect'].map(lambda a: aspect_map.get(a, 'General Experience'))
    gold_df['aspect_conditioned_text'] = 'Aspect: ' + gold_df['canonical_aspect'] + ' [SEP] ' + gold_df['clause']
    
    y_gold = gold_df['gold_sentiment'].map(LABEL_MAP).values
    X_gold = tfidf.transform(gold_df['aspect_conditioned_text'])
    
    gold_bench_results = []
    for name, model in reloaded_models.items():
        g_preds = model.predict(X_gold)
        g_acc = accuracy_score(y_gold, g_preds)
        g_macro_f1 = f1_score(y_gold, g_preds, average='macro')
        g_weighted_f1 = f1_score(y_gold, g_preds, average='weighted')
        g_class_f1 = f1_score(y_gold, g_preds, average=None)
        
        gold_bench_results.append({
            'Model': name,
            'Accuracy (%)': round(g_acc * 100, 2),
            'Macro-F1 (%)': round(g_macro_f1 * 100, 2),
            'Weighted-F1 (%)': round(g_weighted_f1 * 100, 2),
            'Negative F1 (%)': round(g_class_f1[0] * 100, 2),
            'Neutral F1 (%)': round(g_class_f1[1] * 100, 2),
            'Positive F1 (%)': round(g_class_f1[2] * 100, 2)
        })
    
    df_gold_summary = pd.DataFrame(gold_bench_results)
    df_gold_summary.to_csv(os.path.join(OUTPUT_DIR, 'gold_650_benchmark_metrics.csv'), index=False)
    print("\n================= GOLD 650 BENCHMARK METRICS (PREDICTED-SET MODELS) =================")
    print(df_gold_summary.to_string(index=False))
else:
    print("Gold benchmark dataset not found, skipping external evaluation.")